# Qwen2.5-1.5B 한국어 Attention 헤드 분석

기존 예비 실험의 평가 표본을 재사용합니다. ① 1번 묶음의 512토큰으로 레이어를 탐색하고 ② 후보 레이어의 헤드를 하나씩 끈 뒤 ③ 상위 헤드를 **사용하지 않은 2·3번 묶음 전체**로 검증합니다. FP16 모델의 헤드 출력을 `o_proj` 직전에서 0으로 바꾸며 가중치는 수정하지 않습니다.

Colab GPU 런타임에서 위에서 아래로 실행하세요. 업로드 창에서 `pilot_results/pilot_samples.json` 하나를 선택합니다. CSV 결과는 브라우저로 다운로드되며 모델 체크포인트는 저장하지 않습니다. 이 노트북은 GPU 실험을 실행한 결과를 포함하지 않습니다.

`S = (마스킹 후 한국어 PPL / 기준 한국어 PPL - 1) × 100 - (마스킹 후 영어 PPL / 기준 영어 PPL - 1) × 100`입니다. 값이 크고 한국어 PPL 증가율도 양수인 헤드를 후보로 봅니다. 영어 성능은 부작용 확인용입니다. 헤드 마스킹 민감도가 곧 양자화 민감도를 뜻하지는 않습니다.


In [ ]:
%pip install -q "transformers>=4.46,<5"


In [ ]:
import csv
import hashlib
import io
import json
import math
from datetime import datetime, timezone

import torch
import transformers
from google.colab import files
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Colab 런타임을 GPU로 변경하세요.")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
LAYER_CANDIDATES = 6
HEAD_CANDIDATES = 10
SEQ_LEN = 512
FULL_TOKENS = 16 * SEQ_LEN

uploaded = files.upload()
if set(uploaded) != {"pilot_samples.json"}:
    raise ValueError("pilot_results/pilot_samples.json 하나만 업로드하세요.")
sample_bytes = uploaded["pilot_samples.json"]
samples = json.loads(sample_bytes.decode("utf-8"))
if set(samples.get("eval_groups", {})) != {"ko", "en"} or any(
    len(samples["eval_groups"][lang]) != 3 for lang in ("ko", "en")
):
    raise ValueError("예비 실험의 한국어·영어 평가 묶음이 각각 3개여야 합니다.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to("cuda:0").eval()
config = model.config
N_LAYERS = len(model.model.layers)
N_HEADS = config.num_attention_heads
HEAD_DIM = getattr(config, "head_dim", None) or config.hidden_size // N_HEADS
print("GPU:", torch.cuda.get_device_name(0))
print("레이어:", N_LAYERS, "Query 헤드/레이어:", N_HEADS, "헤드 폭:", HEAD_DIM)


In [ ]:
def make_token_group(texts, n_tokens):
    ids = []
    for text in texts:
        ids.extend(tokenizer(text, add_special_tokens=False)["input_ids"])
        if tokenizer.eos_token_id is not None:
            ids.append(tokenizer.eos_token_id)
        if len(ids) >= n_tokens:
            return ids[:n_tokens]
    raise ValueError("평가 묶음의 토큰 수가 부족합니다.")

# 탐색에는 1번 묶음만, 검증에는 2·3번 묶음만 사용합니다.
screen_tokens = {
    lang: make_token_group(samples["eval_groups"][lang][0], SEQ_LEN)
    for lang in ("ko", "en")
}
validation_tokens = {
    (lang, group): make_token_group(
        samples["eval_groups"][lang][group - 1], FULL_TOKENS
    )
    for lang in ("ko", "en") for group in (2, 3)
}

def mean_nll(token_ids, layer=None, head=None):
    hook = None
    if layer is not None:
        projection = model.model.layers[layer].self_attn.o_proj

        def mask(_module, args):
            x = args[0].clone()
            if x.shape[-1] != N_HEADS * HEAD_DIM:
                raise ValueError(f"예상과 다른 attention 출력 폭: {x.shape[-1]}")
            if head is None:
                x.zero_()                  # 레이어 탐색: 모든 Query 헤드 비활성화
            else:
                start = head * HEAD_DIM
                x[..., start:start + HEAD_DIM] = 0
            return (x,)

        hook = projection.register_forward_pre_hook(mask)

    try:
        total_nll = 0.0
        predicted_tokens = 0
        for start in range(0, len(token_ids), SEQ_LEN):
            chunk = token_ids[start:start + SEQ_LEN]
            if len(chunk) != SEQ_LEN:
                raise ValueError("평가 토큰 길이가 512의 배수가 아닙니다.")
            input_ids = torch.tensor([chunk], dtype=torch.long, device="cuda:0")
            with torch.inference_mode():
                output = model(input_ids=input_ids, labels=input_ids)
            count = SEQ_LEN - 1
            total_nll += output.loss.item() * count
            predicted_tokens += count
            del input_ids, output
        return total_nll / predicted_tokens
    finally:
        if hook is not None:
            hook.remove()

def score(token_ids, baseline_nll, layer, head=None):
    masked_nll = mean_nll(token_ids, layer, head)
    return 100 * math.expm1(masked_nll - baseline_nll)

baseline_screen = {lang: mean_nll(ids) for lang, ids in screen_tokens.items()}
print("탐색 기준 PPL:", {
    lang: round(math.exp(loss), 4) for lang, loss in baseline_screen.items()
})
# 훅이 제거된 뒤 기준값이 돌아오는지 확인합니다.
assert abs(mean_nll(screen_tokens["ko"]) - baseline_screen["ko"]) < 1e-3


## 1. 레이어 탐색

1번 묶음의 512토큰에서 레이어 전체를 하나씩 끕니다. `S`가 크고 한국어 PPL이 증가한 레이어 상위 6개를 헤드 탐색 대상으로 고릅니다. 이 단계의 점수는 후보 선정용입니다.


In [ ]:
layer_rows = []
for layer in range(N_LAYERS):
    ko = score(screen_tokens["ko"], baseline_screen["ko"], layer)
    en = score(screen_tokens["en"], baseline_screen["en"], layer)
    layer_rows.append({"layer": layer, "ko_change_pct": ko,
                       "en_change_pct": en, "S_pp": ko - en})

layer_candidates = sorted(
    (row for row in layer_rows if row["ko_change_pct"] > 0),
    key=lambda row: row["S_pp"], reverse=True
)[:LAYER_CANDIDATES]
if not layer_candidates:
    raise RuntimeError("한국어 PPL이 증가한 레이어가 없습니다. 전체 결과를 확인하세요.")
print("헤드를 탐색할 레이어:", [row["layer"] for row in layer_candidates])
display(layer_candidates)


## 2. 후보 레이어의 헤드 탐색

각 Query 헤드를 하나씩 끕니다. Query 헤드 마스킹은 공유 KV 헤드의 가중치를 바꾸지 않습니다.


In [ ]:
head_rows = []
for layer_row in layer_candidates:
    layer = layer_row["layer"]
    for head in range(N_HEADS):
        ko = score(screen_tokens["ko"], baseline_screen["ko"], layer, head)
        en = score(screen_tokens["en"], baseline_screen["en"], layer, head)
        head_rows.append({"layer": layer, "head": head,
                          "ko_change_pct": ko, "en_change_pct": en,
                          "S_pp": ko - en})

head_candidates = sorted(
    (row for row in head_rows if row["ko_change_pct"] > 0),
    key=lambda row: row["S_pp"], reverse=True
)[:HEAD_CANDIDATES]
if not head_candidates:
    raise RuntimeError("한국어 PPL이 증가한 헤드가 없습니다. 탐색 결과를 확인하세요.")
print("검증할 헤드:", [(row["layer"], row["head"]) for row in head_candidates])
display(head_candidates)


## 3. 사용하지 않은 2·3번 묶음으로 검증

후보마다 각 언어·묶음의 8192토큰을 평가합니다. 두 묶음에서 한국어 PPL 증가와 `S`의 방향이 유지되는지 확인합니다. 탐색 표본으로 다시 후보를 고르지 않습니다.


In [ ]:
baseline_validation = {
    key: mean_nll(ids) for key, ids in validation_tokens.items()
}
validation_rows = []
for candidate in head_candidates:
    layer, head = candidate["layer"], candidate["head"]
    for group in (2, 3):
        ko = score(validation_tokens["ko", group],
                   baseline_validation["ko", group], layer, head)
        en = score(validation_tokens["en", group],
                   baseline_validation["en", group], layer, head)
        validation_rows.append({"layer": layer, "head": head, "group": group,
                                "ko_change_pct": ko, "en_change_pct": en,
                                "S_pp": ko - en})

summary_rows = []
for candidate in head_candidates:
    rows = [row for row in validation_rows if
            (row["layer"], row["head"]) ==
            (candidate["layer"], candidate["head"])]
    summary_rows.append({
        "layer": candidate["layer"], "head": candidate["head"],
        "mean_ko_change_pct": sum(row["ko_change_pct"] for row in rows) / 2,
        "mean_en_change_pct": sum(row["en_change_pct"] for row in rows) / 2,
        "mean_S_pp": sum(row["S_pp"] for row in rows) / 2,
        "positive_ko_groups": sum(row["ko_change_pct"] > 0 for row in rows),
        "positive_S_groups": sum(row["S_pp"] > 0 for row in rows),
    })
summary_rows.sort(key=lambda row: row["mean_S_pp"], reverse=True)
display(summary_rows)


## 4. CSV를 브라우저로 다운로드

아래 셀을 실행하면 결과 CSV 4개가 브라우저 다운로드 목록에 나타납니다. 다운로드 완료를 확인한 뒤 런타임을 종료하세요. `head_validation_by_group.csv`의 두 묶음이 같은 방향인지 확인하고, 평균만으로 결론을 내리지 마세요.


In [ ]:
def download_csv(name, rows):
    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
    with open(name, "w", encoding="utf-8-sig", newline="") as f:
        f.write(buffer.getvalue())
    files.download(name)

run_info = [{
    "model_id": MODEL_ID,
    "sample_sha256": hashlib.sha256(sample_bytes).hexdigest(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "screen_group": 1, "screen_tokens_per_language": SEQ_LEN,
    "validation_groups": "2,3", "validation_tokens_per_group": FULL_TOKENS,
    "layer_candidates": LAYER_CANDIDATES, "head_candidates": HEAD_CANDIDATES,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}]
for name, rows in (
    ("head_layer_screen.csv", layer_rows),
    ("head_screen.csv", head_rows),
    ("head_validation_by_group.csv", validation_rows),
    ("head_validation_summary.csv", summary_rows),
    ("head_run_info.csv", run_info),
):
    download_csv(name, rows)
